# Module 03 — Unsupervised & Statistical Learning
## 03-05: Independent Component Analysis

**Objective:** Implement FastICA from scratch using three contrast functions, solve the cocktail party signal-separation problem, and clearly distinguish statistical independence (what ICA achieves) from mere uncorrelatedness (what PCA achieves).

**Prerequisites:** 1-06 (Linear Algebra Foundations)

## Part 0 — Setup & Prerequisites

PCA finds a basis that maximises variance — it decorrelates data but leaves higher-order statistical structure untouched. **Independent Component Analysis (ICA)** goes further: it finds a basis where the projections are statistically *independent*, not merely uncorrelated. This is only possible because independent signals are non-Gaussian (the central limit theorem guarantees that any mixture of independent signals is *more* Gaussian than the originals, so we need to search for the *least* Gaussian directions).

In this notebook we:
1. Demonstrate the difference between **uncorrelatedness** and **statistical independence** with concrete examples.
2. Implement the complete **FastICA preprocessing pipeline** from scratch: centering, PCA whitening, and the fixed-point iteration.
3. Implement three **contrast functions** (logcosh, kurtosis, exponential) that measure non-Gaussianity.
4. Solve the **cocktail party problem**: separate three mixed audio-like signals back into their original sources.
5. Show why **Gaussian sources cannot be separated** by ICA, and measure separation quality with the Amari performance index.

**Prerequisites:** 1-06 (Linear Algebra Foundations)

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import sys
import time
import warnings
import itertools
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sklearn.decomposition import FastICA as SklearnFastICA, PCA as SklearnPCA
from sklearn.preprocessing import StandardScaler

print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}")

In [ ]:
# ── Reproducibility ───────────────────────────────────────────────────────────
import random

SEED = 1103
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
N_SAMPLES    = 2000     # number of time-series samples
DURATION     = 4.0      # seconds of simulated signal
N_SOURCES    = 3        # number of independent sources
NOISE_STD    = 0.05     # additive Gaussian noise level for robustness test
MAX_ITER     = 300      # FastICA maximum iterations per component
TOL          = 1e-7     # FastICA convergence tolerance
K_NEIGHBOURS = 10       # for neighbourhood preservation check

### Data Generation: The Cocktail Party Problem

Imagine $n$ microphones at different positions in a room where $n$ people are speaking simultaneously. Each microphone records a different linear mixture of the voices. ICA's goal is to recover the original voices (independent sources) from only the microphone recordings (observed mixtures), *without knowing* the positions of the microphones or speakers.

Mathematically:
$$\mathbf{X} = \mathbf{A}\,\mathbf{S}$$

where $\mathbf{S} \in \mathbb{R}^{k \times T}$ are the source signals, $\mathbf{A} \in \mathbb{R}^{k \times k}$ is the unknown mixing matrix, and $\mathbf{X} \in \mathbb{R}^{k \times T}$ are the observations. ICA finds $\mathbf{W}$ such that $\hat{\mathbf{S}} = \mathbf{W}\mathbf{X} \approx \mathbf{S}$ (up to permutation and scaling).

In [ ]:
# ── Source signal generators ──────────────────────────────────────────────────
def sine_wave(t: np.ndarray, frequency: float, phase: float = 0.0) -> np.ndarray:
    """Generate a sinusoidal wave."""
    return np.sin(2.0 * np.pi * frequency * t + phase)


def sawtooth_wave(t: np.ndarray, frequency: float) -> np.ndarray:
    """Generate a sawtooth wave with values in [-1, 1)."""
    phase = (t * frequency) % 1.0   # fractional cycle position in [0, 1)
    return 2.0 * phase - 1.0        # remap to [-1, 1)


def square_wave(t: np.ndarray, frequency: float) -> np.ndarray:
    """Generate a square wave with values in {-1, +1}."""
    return np.sign(np.sin(2.0 * np.pi * frequency * t))


def generate_source_signals(
    n_samples: int = N_SAMPLES,
    duration: float = DURATION,
    random_state: int = SEED,
) -> tuple[np.ndarray, np.ndarray]:
    """Generate three independent, non-Gaussian source signals.

    Creates a sinusoid (3 Hz), sawtooth (1.5 Hz), and square wave (5 Hz).
    All sources are zero-mean and unit-variance.

    Args:
        n_samples: Number of discrete time samples.
        duration: Signal duration in seconds.
        random_state: Seed for reproducibility.

    Returns:
        Tuple of:
            - S: Source matrix of shape (3, n_samples).
            - t: Time vector of shape (n_samples,).
    """
    t = np.linspace(0.0, duration, n_samples, endpoint=False)

    s1 = sine_wave(t, frequency=3.0)       # smooth, sub-Gaussian
    s2 = sawtooth_wave(t, frequency=1.5)   # linearly ramping, sub-Gaussian
    s3 = square_wave(t, frequency=5.0)     # binary spike, super-Gaussian

    S = np.vstack([s1, s2, s3]).astype(np.float64)
    # Standardise each source to zero mean and unit variance
    S = S - S.mean(axis=1, keepdims=True)
    S = S / S.std(axis=1, keepdims=True)
    return S, t


S_true, t_signal = generate_source_signals()
print(f"Source matrix shape: {S_true.shape}")
print(f"Source means:  {S_true.mean(axis=1).round(6)}  (should be ≈ 0)")
print(f"Source stds:   {S_true.std(axis=1).round(6)}  (should be ≈ 1)")

In [ ]:
# ── Mixing matrix and observed signals ────────────────────────────────────────
def create_mixing_matrix(
    n_sources: int = N_SOURCES,
    condition_number: float = 4.0,
    random_state: int = SEED,
) -> np.ndarray:
    """Generate a random, well-conditioned square mixing matrix.

    Constructs A from a random SVD with controlled singular values so
    the matrix is invertible and not ill-conditioned (which would make
    separation numerically unstable).

    Args:
        n_sources: Dimension of the square matrix.
        condition_number: Ratio of largest to smallest singular value.
        random_state: Seed for reproducibility.

    Returns:
        Mixing matrix A of shape (n_sources, n_sources).
    """
    rng = np.random.RandomState(random_state)
    A_rand = rng.randn(n_sources, n_sources)
    U, _, Vt = np.linalg.svd(A_rand)               # random orthogonal frames
    s_vals = np.linspace(1.0, condition_number, n_sources)  # controlled spectrum
    A = U @ np.diag(s_vals) @ Vt
    return A


def mix_signals(
    S: np.ndarray,
    A: np.ndarray | None = None,
    noise_std: float = 0.0,
    random_state: int = SEED,
) -> tuple[np.ndarray, np.ndarray]:
    """Linearly mix source signals: X = A @ S + noise.

    Args:
        S: Source matrix of shape (n_sources, n_samples).
        A: Mixing matrix of shape (n_sources, n_sources).
            If None, a random well-conditioned matrix is created.
        noise_std: Standard deviation of additive Gaussian noise.
        random_state: Seed for the noise generator.

    Returns:
        Tuple of (X, A) where X is the observed signal matrix.
    """
    rng = np.random.RandomState(random_state)
    if A is None:
        A = create_mixing_matrix(n_sources=S.shape[0], random_state=random_state)
    X = A @ S
    if noise_std > 0.0:
        X = X + rng.randn(*X.shape) * noise_std
    return X, A


A_true = create_mixing_matrix()
X_obs, _ = mix_signals(S_true, A=A_true)
print(f"Mixing matrix A:\n{A_true.round(3)}")
print(f"\nObserved signal shape: {X_obs.shape}")
print(f"Observed means: {X_obs.mean(axis=1).round(4)}")

In [ ]:
# ── EDA: source signals and their mixtures ────────────────────────────────────
SOURCE_NAMES = ["Sine (3 Hz)", "Sawtooth (1.5 Hz)", "Square (5 Hz)"]
OBS_NAMES    = [f"Mixture {i+1}" for i in range(N_SOURCES)]
t_plot       = t_signal[:500]   # first 1 second

fig, axes = plt.subplots(N_SOURCES, 2, figsize=(14, 6), sharex=True)
fig.suptitle("Cocktail party: source signals (left) and their linear mixtures (right)",
             fontsize=12)

colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]
for i in range(N_SOURCES):
    axes[i, 0].plot(t_plot, S_true[i, :500], color=colors[i], linewidth=1.2)
    axes[i, 0].set_ylabel(SOURCE_NAMES[i], fontsize=9)
    axes[i, 0].grid(True, alpha=0.3)

    axes[i, 1].plot(t_plot, X_obs[i, :500], color="gray", linewidth=1.0)
    axes[i, 1].set_ylabel(OBS_NAMES[i], fontsize=9)
    axes[i, 1].grid(True, alpha=0.3)

axes[-1, 0].set_xlabel("Time (s)")
axes[-1, 1].set_xlabel("Time (s)")
axes[0, 0].set_title("True sources (unknown to ICA)")
axes[0, 1].set_title("Observed mixtures (input to ICA)")
plt.tight_layout()
plt.show()

# Correlation matrix of mixtures: show they are correlated
corr_obs = np.corrcoef(X_obs)
print("Correlation matrix of observations (non-diagonal → correlated):")
print(corr_obs.round(3))

---
## Part 1 — ICA Theory and From-Scratch Implementation

### 1.1 Independence vs Uncorrelatedness

**Uncorrelatedness** is a second-order property: $\text{Cov}(X, Y) = 0$, or equivalently $\mathbb{E}[XY] = \mathbb{E}[X]\,\mathbb{E}[Y]$.

**Statistical independence** is a much stronger property: $p(X, Y) = p(X)\,p(Y)$, i.e., knowing $X$ tells you *nothing* about $Y$. Independence implies uncorrelatedness (for any function $f$, $\mathbb{E}[f(X)g(Y)] = \mathbb{E}[f(X)]\mathbb{E}[g(Y)]$), but not vice versa.

PCA only decorrelates (removes second-order dependencies). ICA removes all higher-order statistical dependencies. The classic counter-example: $X \sim \mathcal{U}(-1,1)$ and $Y = X^2$ are **uncorrelated** (by symmetry, $\mathbb{E}[X \cdot X^2] = \mathbb{E}[X^3] = 0$) yet **completely dependent** (knowing $X$ perfectly determines $Y$).

In [ ]:
# ── Demo: uncorrelated yet dependent ─────────────────────────────────────────
rng_demo = np.random.RandomState(SEED)
n_demo   = 5000

X_demo = rng_demo.uniform(-1.0, 1.0, n_demo)
Y_demo = X_demo ** 2

covariance  = float(np.cov(X_demo, Y_demo)[0, 1])
pearson_r   = float(np.corrcoef(X_demo, Y_demo)[0, 1])
mutual_info_approx = float(-0.5 * np.log(1.0 - pearson_r**2 + 1e-12))  # approx for Gaussian

# Check whether knowing X reduces uncertainty about Y
# If X ∈ (-0.1, 0.1), Y ∈ (0, 0.01) — very constrained; not independent!
mask_near_zero = np.abs(X_demo) < 0.1
y_cond_range = Y_demo[mask_near_zero].ptp()  # range when X ≈ 0
y_uncond_range = Y_demo.ptp()                # unconditional range

print("Uncorrelated-but-dependent example: Y = X²")
print(f"  Cov(X, Y)     = {covariance:.6f}  (≈ 0 → uncorrelated)")
print(f"  Pearson r     = {pearson_r:.6f}")
print(f"  Y range (unconditional): {y_uncond_range:.4f}")
print(f"  Y range given |X| < 0.1: {y_cond_range:.4f}  (much smaller → X informs Y!)")

# Another test: E[X²Y²] ≠ E[X²] E[Y²] if dependent
e_x2y2     = float(np.mean(X_demo**2 * Y_demo**2))
e_x2_e_y2  = float(np.mean(X_demo**2) * np.mean(Y_demo**2))
print(f"\n  E[X²·Y²] = {e_x2y2:.6f}")
print(f"  E[X²]·E[Y²] = {e_x2_e_y2:.6f}  (differ → higher-order dependence!)")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(X_demo[:1000], Y_demo[:1000], s=4, alpha=0.4, color="steelblue")
axes[0].set_xlabel("X ~ Uniform(-1, 1)")
axes[0].set_ylabel("Y = X²")
axes[0].set_title(f"Uncorrelated (r={pearson_r:.4f}) but dependent")
axes[0].grid(True, alpha=0.3)

# PCA on (X, Y) — whitens and decorrelates but cannot separate
XY_matrix = np.vstack([X_demo, Y_demo])
pca_demo = SklearnPCA(n_components=2)
XY_pca = pca_demo.fit_transform(XY_matrix.T).T
r_pca = float(np.corrcoef(XY_pca[0], XY_pca[1])[0, 1])
axes[1].scatter(XY_pca[0, :1000], XY_pca[1, :1000], s=4, alpha=0.4, color="darkorange")
axes[1].set_xlabel("PCA Component 1")
axes[1].set_ylabel("PCA Component 2")
axes[1].set_title(f"After PCA (r={r_pca:.4f}): decorrelated but still dependent")
axes[1].grid(True, alpha=0.3)

plt.suptitle("PCA removes correlation but not statistical dependence", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── Super-Gaussian vs Sub-Gaussian: PDF shapes and kurtosis sign ──────────────
# The sign of excess kurtosis determines whether a source is
# super-Gaussian (positive kurtosis, heavy tails, e.g. speech, spikes)
# or sub-Gaussian (negative kurtosis, light tails, e.g. uniform, sinusoid).
# ICA works for both, but different contrast functions are more efficient
# depending on the source type.

def estimate_pdf_hist(
    x: np.ndarray,
    n_bins: int = 60,
    density: bool = True,
) -> tuple[np.ndarray, np.ndarray]:
    """Estimate a probability density histogram.

    Args:
        x: 1-D sample array.
        n_bins: Number of histogram bins.
        density: If True, normalise to form a probability density.

    Returns:
        Tuple of (bin_centres, densities).
    """
    counts, bin_edges = np.histogram(x, bins=n_bins, density=density)
    bin_centres = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    return bin_centres, counts


def gaussian_pdf(x: np.ndarray, mu: float = 0.0, sigma: float = 1.0) -> np.ndarray:
    """Evaluate the Gaussian PDF N(mu, sigma^2) at points x."""
    return (1.0 / (sigma * np.sqrt(2.0 * np.pi))) * np.exp(-0.5 * ((x - mu) / sigma) ** 2)


# Standardised source signals and Gaussian reference
x_gauss_ref = np.random.RandomState(SEED).randn(N_SAMPLES)
sources_for_pdf = [
    (S_true[0], "Sine (sub-Gaussian)",    "#1f77b4"),
    (S_true[1], "Sawtooth (sub-Gaussian)", "#ff7f0e"),
    (S_true[2], "Square (super-Gaussian)", "#2ca02c"),
    (x_gauss_ref, "Gaussian reference",   "#d62728"),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=False)
fig.suptitle("PDF shapes: sub-Gaussian (light tails) vs super-Gaussian (heavy tails)", fontsize=11)
x_range = np.linspace(-4.5, 4.5, 200)

for ax, (signal, label, color) in zip(axes, sources_for_pdf):
    # Standardise for fair comparison
    s_std = (signal - signal.mean()) / signal.std()
    centres, density = estimate_pdf_hist(s_std)
    kurt = compute_kurtosis(s_std)

    ax.bar(centres, density, width=(centres[1] - centres[0]), alpha=0.6,
           color=color, label=label)
    ax.plot(x_range, gaussian_pdf(x_range), color="black", linestyle="--",
            linewidth=1.5, label="Gaussian")
    ax.set_title(f"{label}\nkurtosis = {kurt:+.2f}", fontsize=8)
    ax.set_xlabel("Standardised value")
    ax.set_xlim(-4.5, 4.5)
    ax.grid(True, alpha=0.25)
    if ax == axes[0]:
        ax.set_ylabel("Density")

plt.tight_layout()
plt.show()

print("Sub-Gaussian (kurt < 0): distribution is more uniform / bounded than Gaussian")
print("Super-Gaussian (kurt > 0): distribution has heavier tails / sharper peak than Gaussian")
print("Gaussian (kurt ≈ 0):      the reference — ICA has no gradient to use")

### 1.2 Preprocessing: Centering and Whitening

FastICA requires two preprocessing steps:

1. **Centering:** Subtract the row mean so $\mathbb{E}[\mathbf{x}] = 0$. Without this, the first ICA component would capture the mean rather than a source.

2. **Whitening (sphering):** Transform the data so its covariance is the identity matrix, $\mathbb{E}[\tilde{\mathbf{x}}\tilde{\mathbf{x}}^\top] = \mathbf{I}$. This:
   - Removes second-order structure, so FastICA only needs to find *higher-order* non-Gaussianity.
   - Reduces the search from a general unmixing matrix to an **orthogonal** rotation matrix (since rotating an already-white signal keeps it white).

Whitening via eigendecomposition:
$$\mathbf{\Sigma} = \mathbf{E}\,\mathbf{D}\,\mathbf{E}^\top, \quad \mathbf{V} = \mathbf{D}^{-1/2}\mathbf{E}^\top, \quad \tilde{\mathbf{X}} = \mathbf{V}\mathbf{X}_c$$

In [ ]:
def center_signals(
    X: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Subtract the row-wise mean from each signal.

    Args:
        X: Signal matrix of shape (n_signals, n_samples).

    Returns:
        Tuple of (X_centered, mean_vector) where mean_vector has shape
        (n_signals, 1). Store mean_vector to invert centering later.
    """
    mean_vec = X.mean(axis=1, keepdims=True)   # (n_signals, 1)
    return X - mean_vec, mean_vec


def whiten_signals(
    X: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Apply PCA whitening to impose identity covariance.

    Computes V = D^{-1/2} E^T where Sigma = E D E^T is the
    eigendecomposition of the sample covariance of X.  The returned
    whitened data satisfies Cov(X_white) ≈ I.

    Args:
        X: Centered signal matrix of shape (n_signals, n_samples).

    Returns:
        Tuple of:
            - X_white: Whitened data, shape (n_signals, n_samples).
            - whitening_matrix: V = D^{-1/2} E^T, shape (n_signals, n_signals).
            - dewhitening_matrix: V^{-1} = E D^{1/2}, shape (n_signals, n_signals).
            - eigenvalues: Eigenvalues of the covariance, shape (n_signals,).
    """
    n_samples = X.shape[1]
    # Sample covariance: (1/n) X X^T
    cov = (X @ X.T) / n_samples

    # eigh: guaranteed real eigenvalues for symmetric matrices
    eigenvalues, eigenvectors = np.linalg.eigh(cov)     # ascending order

    # Sort descending (largest variance first — consistent with PCA convention)
    sorted_idx   = np.argsort(eigenvalues)[::-1]
    eigenvalues  = eigenvalues[sorted_idx]
    eigenvectors = eigenvectors[:, sorted_idx]

    # Whitening and de-whitening matrices
    D_inv_sqrt       = np.diag(eigenvalues ** (-0.5))
    D_sqrt           = np.diag(eigenvalues ** 0.5)
    whitening_matrix   = D_inv_sqrt @ eigenvectors.T   # (d, d)
    dewhitening_matrix = eigenvectors @ D_sqrt         # (d, d)

    X_white = whitening_matrix @ X
    return X_white, whitening_matrix, dewhitening_matrix, eigenvalues


# Apply to observed data
X_centered, X_mean = center_signals(X_obs)
X_white, V, V_inv, eigvals = whiten_signals(X_centered)
print(f"Whitened shape: {X_white.shape}")
print(f"Eigenvalues (variances): {eigvals.round(4)}")

### Verifying the Whitening Transform

After whitening, the covariance matrix should be the identity: all diagonal entries ≈ 1, all off-diagonal entries ≈ 0. We visualise this as a heat-map and report the maximum deviation from the identity to confirm correctness numerically.

In [ ]:
# ── Verify whitening: covariance should be ≈ identity ─────────────────────────
cov_before = np.cov(X_centered)
cov_after  = np.cov(X_white)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

im0 = axes[0].imshow(cov_before, cmap="coolwarm", vmin=-2, vmax=2)
axes[0].set_title("Covariance before whitening")
plt.colorbar(im0, ax=axes[0])
axes[0].set_xticks(range(N_SOURCES))
axes[0].set_yticks(range(N_SOURCES))

im1 = axes[1].imshow(cov_after, cmap="coolwarm", vmin=-0.1, vmax=1.1)
axes[1].set_title("Covariance after whitening (≈ identity)")
plt.colorbar(im1, ax=axes[1])
axes[1].set_xticks(range(N_SOURCES))
axes[1].set_yticks(range(N_SOURCES))

plt.suptitle("Effect of whitening on the covariance matrix", fontsize=11)
plt.tight_layout()
plt.show()

max_off_diag_after = np.abs(cov_after - np.eye(N_SOURCES)).max()
print(f"Max off-diagonal deviation after whitening: {max_off_diag_after:.2e}  (should be tiny)")

### 1.3 Contrast Functions: Measuring Non-Gaussianity

The heart of ICA is finding projections that are maximally non-Gaussian. We measure non-Gaussianity via **negentropy**:
$$J(\mathbf{w}) \approx \left[\mathbb{E}\{G(\mathbf{w}^\top \tilde{\mathbf{x}})\} - \mathbb{E}\{G(\nu)\}\right]^2$$

where $\nu \sim \mathcal{N}(0, 1)$ is a standard Gaussian and $G$ is a **contrast function** chosen to be smooth and non-polynomial. Maximising $J$ finds the most non-Gaussian direction.

The fixed-point update uses $g = G'$ and $g' = G''$:
$$\mathbf{w}^+ = \mathbb{E}\{\tilde{\mathbf{x}}\,g(\mathbf{w}^\top \tilde{\mathbf{x}})\} - \mathbb{E}\{g'(\mathbf{w}^\top \tilde{\mathbf{x}})\}\,\mathbf{w}$$

| Contrast | $G(u)$ | $g(u) = G'(u)$ | Best for |
|----------|--------|----------------|----------|
| **logcosh** | $\frac{1}{a}\log\cosh(au)$ | $\tanh(au)$ | General-purpose; robust |
| **kurtosis** | $\frac{u^4}{4}$ | $u^3$ | Super-Gaussian; outlier-sensitive |
| **exp** | $-e^{-u^2/2}$ | $u\,e^{-u^2/2}$ | Super-Gaussian; less outlier-sensitive |

In [ ]:
def logcosh_contrast(
    u: np.ndarray,
    alpha: float = 1.0,
) -> tuple[np.ndarray, float]:
    """Logcosh contrast function G(u) = (1/alpha)*log(cosh(alpha*u)).

    Most robust general-purpose contrast function.  Bounded derivative
    means it is insensitive to outliers.

    Args:
        u: Projected sample vector of shape (n_samples,).
        alpha: Scale parameter, typically in [1, 2].

    Returns:
        Tuple of (g_vals, g_prime_mean) where:
            - g_vals: tanh(alpha*u), shape (n_samples,).
            - g_prime_mean: scalar mean of g'(u) = alpha*(1 - tanh^2(alpha*u)).
    """
    tanh_val     = np.tanh(alpha * u)
    g_vals       = tanh_val
    g_prime_mean = float(np.mean(alpha * (1.0 - tanh_val ** 2)))
    return g_vals, g_prime_mean


def kurtosis_contrast(u: np.ndarray) -> tuple[np.ndarray, float]:
    """Kurtosis contrast function G(u) = u^4/4.

    Maximising this is equivalent to maximising the (excess) kurtosis,
    the fourth standardised cumulant.  Fast but sensitive to outliers.

    Args:
        u: Projected sample vector of shape (n_samples,).

    Returns:
        Tuple of (g_vals, g_prime_mean) where:
            - g_vals: u^3, shape (n_samples,).
            - g_prime_mean: scalar mean of g'(u) = 3*u^2.
    """
    g_vals       = u ** 3
    g_prime_mean = float(np.mean(3.0 * u ** 2))
    return g_vals, g_prime_mean


def exp_contrast(u: np.ndarray) -> tuple[np.ndarray, float]:
    """Exponential contrast function G(u) = -exp(-u^2/2).

    Good for super-Gaussian (heavy-tailed) sources like audio.  The
    exponential dampening makes it more robust to outliers than kurtosis.

    Args:
        u: Projected sample vector of shape (n_samples,).

    Returns:
        Tuple of (g_vals, g_prime_mean) where:
            - g_vals: u*exp(-u^2/2), shape (n_samples,).
            - g_prime_mean: scalar mean of g'(u) = (1-u^2)*exp(-u^2/2).
    """
    exp_val      = np.exp(-0.5 * u ** 2)
    g_vals       = u * exp_val
    g_prime_mean = float(np.mean((1.0 - u ** 2) * exp_val))
    return g_vals, g_prime_mean


CONTRAST_FNS: dict[str, callable] = {
    "logcosh":  logcosh_contrast,
    "kurtosis": kurtosis_contrast,
    "exp":      exp_contrast,
}

### Visualising Contrast Functions

Plotting $G$, $g$, and $g'$ side by side clarifies the key differences between the three contrast functions. Notice that logcosh's derivative $\tanh(u)$ saturates to $\pm 1$ for large $|u|$ — this bounded response makes it robust to outlier projections. By contrast, the kurtosis derivative $u^3$ grows without bound, causing instability if the data has heavy tails or outliers.

In [ ]:
# ── Visualise contrast functions and their derivatives ────────────────────────
u_range = np.linspace(-4.0, 4.0, 400)

G_logcosh  = np.log(np.cosh(u_range))
G_kurtosis = u_range ** 4 / 4.0
G_exp      = -np.exp(-0.5 * u_range ** 2)

g_logcosh  = np.tanh(u_range)
g_kurtosis = u_range ** 3
g_exp      = u_range * np.exp(-0.5 * u_range ** 2)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(u_range, G_logcosh / G_logcosh.max(),   label="logcosh (normalised)",  linewidth=2)
axes[0].plot(u_range, G_kurtosis / G_kurtosis.max(), label="kurtosis (normalised)", linewidth=2, linestyle="--")
axes[0].plot(u_range, (G_exp - G_exp.min()) / (G_exp.max() - G_exp.min()),
             label="exp (normalised)", linewidth=2, linestyle=":")
axes[0].set_title("Contrast functions G(u)")
axes[0].set_xlabel("u")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(u_range, g_logcosh,  label="tanh(u) [logcosh]",       linewidth=2)
axes[1].plot(u_range, np.clip(g_kurtosis, -5, 5),
             label="u³ [kurtosis] (clipped)", linewidth=2, linestyle="--")
axes[1].plot(u_range, g_exp,      label="u·exp(−u²/2) [exp]",      linewidth=2, linestyle=":")
axes[1].set_title("Derivatives g(u) = G'(u)")
axes[1].set_xlabel("u")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(-4, 4)

plt.suptitle("ICA contrast functions — logcosh is smoothest and most robust", fontsize=11)
plt.tight_layout()
plt.show()

### 1.4 FastICA Fixed-Point Iteration

FastICA uses **Newton's method** to maximise non-Gaussianity.  Given whitened data $\tilde{\mathbf{X}} \in \mathbb{R}^{d \times T}$, we seek unit vectors $\mathbf{w}$ that maximise $J(\mathbf{w})$.  The fixed-point update is:

$$\mathbf{w}^+ = \frac{1}{T}\tilde{\mathbf{X}}\,g(\mathbf{w}^\top \tilde{\mathbf{X}})^\top - \mathbb{E}\{g'(\mathbf{w}^\top \tilde{\mathbf{x}})\}\,\mathbf{w}, \quad \mathbf{w} \leftarrow \frac{\mathbf{w}^+}{\|\mathbf{w}^+\|}$$

Convergence is checked by $|1 - |\mathbf{w}_\text{new}^\top \mathbf{w}_\text{old}|| < \epsilon$ (the angle between consecutive iterates vanishes).

**Deflation** extracts components one by one: after finding $\mathbf{w}_1, \ldots, \mathbf{w}_{k-1}$, we Gram-Schmidt-orthogonalise $\mathbf{w}_k$ against all previous components before normalising.  This ensures the $k$ components are mutually orthogonal (and hence uncorrelated, since the data is already whitened).

In [ ]:
def fastica_one_component(
    X_white: np.ndarray,
    contrast_fn: callable,
    max_iter: int = MAX_ITER,
    tol: float = TOL,
    init_w: np.ndarray | None = None,
    random_state: int | None = None,
) -> tuple[np.ndarray, int, list[float]]:
    """Extract one independent component via FastICA fixed-point iteration.

    Finds the unit vector w in R^d that maximises non-Gaussianity of
    w^T X_white, as measured by the contrast function G.

    Fixed-point update (vectorised form):
        w+ = (1/T) X_white @ g(w^T X_white)^T  -  E[g'(w^T x)] * w
        w  = w+ / ||w+||

    Args:
        X_white: Whitened data, shape (n_signals, n_samples).
        contrast_fn: Callable (u: ndarray) -> (g_vals: ndarray, g_prime_mean: float).
        max_iter: Maximum number of iterations.
        tol: Convergence threshold for |1 - |w_new · w_old||.
        init_w: Optional initial weight vector of shape (n_signals,).
        random_state: Seed for random initialisation when init_w is None.

    Returns:
        Tuple of:
            - w: Converged weight vector, unit norm, shape (n_signals,).
            - n_iter: Number of iterations until convergence.
            - delta_history: |1 - |w_new · w_old|| per iteration.
    """
    rng      = np.random.RandomState(random_state)
    n_signals, n_samples = X_white.shape

    # Initialise
    if init_w is not None:
        w = init_w.astype(np.float64).copy()
    else:
        w = rng.randn(n_signals).astype(np.float64)
    w /= np.linalg.norm(w)

    delta_history: list[float] = []

    for iteration in range(max_iter):
        w_old = w.copy()

        # Projections of every sample onto current direction w
        u = w @ X_white                          # (n_samples,)

        # Contrast function: g values and E[g']
        g_vals, g_prime_mean = contrast_fn(u)

        # Fixed-point update
        w_new = (X_white @ g_vals) / n_samples - g_prime_mean * w

        # Normalise
        norm = np.linalg.norm(w_new)
        if norm < 1e-15:
            # Degenerate direction: reinitialise randomly
            w_new = rng.randn(n_signals).astype(np.float64)
        w = w_new / np.linalg.norm(w_new)

        # Convergence check: angle between consecutive iterates
        delta = float(abs(1.0 - abs(float(w @ w_old))))
        delta_history.append(delta)

        if delta < tol:
            break

    return w, iteration + 1, delta_history

In [ ]:
def fastica_deflation(
    X_white: np.ndarray,
    n_components: int,
    contrast_fn: callable,
    max_iter: int = MAX_ITER,
    tol: float = TOL,
    random_state: int = SEED,
) -> tuple[np.ndarray, list[int], list[list[float]]]:
    """Extract n_components independent components via deflation.

    Extracts components sequentially. Each new component is orthogonalised
    (Gram-Schmidt) against all previously extracted components to ensure
    they remain distinct and uncorrelated.

    Args:
        X_white: Whitened data, shape (n_signals, n_samples).
        n_components: Number of independent components to extract.
        contrast_fn: Contrast function callable.
        max_iter: Maximum iterations per component.
        tol: Convergence tolerance.
        random_state: Seed for reproducibility.

    Returns:
        Tuple of:
            - W: Unmixing matrix of shape (n_components, n_signals).
              Each row is one extracted weight vector.
            - iters_per_component: Iterations taken for each component.
            - convergence_histories: Delta-history list for each component.
    """
    rng        = np.random.RandomState(random_state)
    n_signals  = X_white.shape[0]
    n_components = min(n_components, n_signals)

    W_rows: list[np.ndarray]          = []
    iters_per_component: list[int]    = []
    conv_histories: list[list[float]] = []

    for k in range(n_components):
        init_w = rng.randn(n_signals).astype(np.float64)
        init_w /= np.linalg.norm(init_w)

        w, n_iter, history = fastica_one_component(
            X_white, contrast_fn,
            max_iter=max_iter, tol=tol,
            init_w=init_w,
        )

        # Gram-Schmidt orthogonalisation against already-found components
        for w_prev in W_rows:
            w = w - float(w @ w_prev) * w_prev

        norm = np.linalg.norm(w)
        if norm < 1e-10:
            # Fallback: random vector orthogonal to all previous ones
            w = rng.randn(n_signals).astype(np.float64)
            for w_prev in W_rows:
                w = w - float(w @ w_prev) * w_prev
            norm = np.linalg.norm(w)
        w = w / norm

        W_rows.append(w)
        iters_per_component.append(n_iter)
        conv_histories.append(history)
        print(f"  Component {k+1}/{n_components}: converged in {n_iter} iterations")

    return np.vstack(W_rows), iters_per_component, conv_histories

---
## Part 2 — Assembling the FastICA Class

We now wrap the preprocessing and deflation steps into a single reusable class with a sklearn-compatible interface.  The overall flow is:

1. `fit(X)`: Center → Whiten → Deflation → store unmixing matrix.
2. `transform(X)`: Apply stored centering + whitening + unmixing.
3. `fit_transform(X)`: Combines both.

The total unmixing applied to *original* (un-centered) data is: $\hat{\mathbf{S}} = \mathbf{W}\,\mathbf{V}\,(\mathbf{X} - \boldsymbol{\mu})$ where $\mathbf{W}$ are the ICA directions in whitened space and $\mathbf{V}$ is the whitening matrix.

In [ ]:
class FastICA:
    """Independent Component Analysis via FastICA fixed-point algorithm.

    Finds a linear unmixing matrix W such that W @ X produces statistically
    independent components. Uses deflation (sequential extraction) with
    Gram-Schmidt orthogonalisation.

    Attributes:
        n_components: Number of independent components to extract.
        contrast: Name of the contrast function ('logcosh', 'kurtosis', 'exp').
        max_iter: Maximum iterations per component in the fixed-point loop.
        tol: Convergence tolerance for the fixed-point iteration.
        random_state: Seed for reproducible random initialisations.
        components_: Unmixing directions in whitened space, shape (k, d).
        whitening_matrix_: V = D^{-1/2} E^T used for preprocessing.
        dewhitening_matrix_: V^{-1} = E D^{1/2}.
        mean_: Sample mean subtracted during centering, shape (d, 1).
        n_iter_: Iterations per component.
        convergence_histories_: Delta-convergence curves per component.
    """

    def __init__(
        self,
        n_components: int = N_SOURCES,
        contrast: str = "logcosh",
        max_iter: int = MAX_ITER,
        tol: float = TOL,
        random_state: int = SEED,
    ) -> None:
        """Initialise FastICA."""
        if contrast not in CONTRAST_FNS:
            raise ValueError(f"contrast must be one of {list(CONTRAST_FNS)}")
        self.n_components  = n_components
        self.contrast      = contrast
        self.max_iter      = max_iter
        self.tol           = tol
        self.random_state  = random_state

        # Fitted attributes (set by fit())
        self.components_: np.ndarray | None           = None
        self.whitening_matrix_: np.ndarray | None     = None
        self.dewhitening_matrix_: np.ndarray | None   = None
        self.mean_: np.ndarray | None                 = None
        self.n_iter_: list[int]                       = []
        self.convergence_histories_: list[list[float]] = []

    def fit(self, X: np.ndarray) -> "FastICA":
        """Fit the ICA model to observations X.

        Args:
            X: Observed signal matrix of shape (n_signals, n_samples).
                Rows are channels; columns are time samples.

        Returns:
            Self (for method chaining).
        """
        contrast_fn = CONTRAST_FNS[self.contrast]

        # Step 1: centre
        X_c, self.mean_ = center_signals(X)

        # Step 2: whiten
        X_white, self.whitening_matrix_, self.dewhitening_matrix_, _ = whiten_signals(X_c)

        # Step 3: sequential fixed-point extraction
        W, self.n_iter_, self.convergence_histories_ = fastica_deflation(
            X_white, self.n_components, contrast_fn,
            max_iter=self.max_iter, tol=self.tol,
            random_state=self.random_state,
        )
        self.components_ = W          # shape (k, d_whitened)
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        """Apply the fitted unmixing matrix to new observations.

        Args:
            X: Signal matrix of shape (n_signals, n_samples).

        Returns:
            Estimated source signals, shape (n_components, n_samples).
        """
        if self.components_ is None:
            raise RuntimeError("Call fit() before transform().")
        X_c     = X - self.mean_
        X_white = self.whitening_matrix_ @ X_c
        return self.components_ @ X_white

    def fit_transform(self, X: np.ndarray) -> np.ndarray:
        """Fit model and return separated source signals.

        Args:
            X: Observed signal matrix of shape (n_signals, n_samples).

        Returns:
            Estimated source signals, shape (n_components, n_samples).
        """
        self.fit(X)
        return self.transform(X)

### Sanity Check on a Simple 2-Source Problem

Before tackling the full 3-channel cocktail party, we validate FastICA on a 2-source problem where both the true sources and mixing matrix are fully known. Absolute Pearson correlation between each recovered component and each true source should be close to 1 for the best-matching pair.

In [ ]:
# ── Sanity check on a 2-source toy problem ─────────────────────────────────────
rng_toy = np.random.RandomState(SEED)
n_toy   = 1000

# Two independent uniform sources (platykurtic / sub-Gaussian)
s_toy1 = rng_toy.uniform(-np.sqrt(3), np.sqrt(3), n_toy)  # unit-variance uniform
s_toy2 = np.sin(2.0 * np.pi * 2.0 * np.linspace(0, 2, n_toy))  # sinusoid
S_toy  = np.vstack([s_toy1, s_toy2])

A_toy = np.array([[1.0, 0.8], [0.5, 1.0]])
X_toy = A_toy @ S_toy

print("Toy 2-source sanity check:")
ica_toy = FastICA(n_components=2, contrast="logcosh", random_state=SEED)
S_hat_toy = ica_toy.fit_transform(X_toy)

# Absolute correlation with true sources (best permutation)
corr_matrix = np.abs(np.corrcoef(S_toy, S_hat_toy)[:2, 2:])
print(f"Absolute correlation matrix (rows=true, cols=recovered):\n{corr_matrix.round(4)}")

fig, axes = plt.subplots(2, 3, figsize=(13, 4), sharex=True)
labels_toy = [["True s₁", "True s₂"], ["Mixed x₁", "Mixed x₂"], ["ICA ŝ₁", "ICA ŝ₂"]]
data_toy   = [S_toy, X_toy, S_hat_toy]
for col, (data, lbls) in enumerate(zip(data_toy, labels_toy)):
    for row in range(2):
        axes[row, col].plot(data[row, :200], linewidth=1.0)
        axes[row, col].set_ylabel(lbls[row], fontsize=9)
        axes[row, col].grid(True, alpha=0.3)

axes[0, 0].set_title("True sources")
axes[0, 1].set_title("Mixed observations")
axes[0, 2].set_title("FastICA recovery (from scratch)")
axes[-1, 0].set_xlabel("Sample")
axes[-1, 1].set_xlabel("Sample")
axes[-1, 2].set_xlabel("Sample")
plt.suptitle("FastICA sanity check — 2-source toy problem", fontsize=11)
plt.tight_layout()
plt.show()

---
## Part 3 — Cocktail Party Problem on 3 Sources

We now apply our from-scratch FastICA and compare against sklearn's implementation and PCA on the 3-source, 3-channel cocktail party problem.

In [ ]:
# ── From-scratch FastICA on the 3-source problem ──────────────────────────────
print("Running from-scratch FastICA (logcosh contrast)...")
t0 = time.time()
ica_scratch = FastICA(n_components=N_SOURCES, contrast="logcosh", random_state=SEED)
S_hat_scratch = ica_scratch.fit_transform(X_obs)
print(f"Done in {time.time()-t0:.2f}s — iterations: {ica_scratch.n_iter_}")

# ── sklearn FastICA for comparison ────────────────────────────────────────────
print("\nRunning sklearn FastICA...")
t0 = time.time()
ica_sklearn = SklearnFastICA(
    n_components=N_SOURCES,
    fun="logcosh",
    max_iter=MAX_ITER,
    tol=1e-4,
    random_state=SEED,
)
# sklearn FastICA expects (n_samples, n_features): transpose convention
S_hat_sklearn = ica_sklearn.fit_transform(X_obs.T).T   # back to (sources, samples)
print(f"Done in {time.time()-t0:.2f}s")

# ── PCA for comparison (decorrelates but does not separate) ───────────────────
print("\nRunning PCA whitening (decorrelation only)...")
pca_sep = SklearnPCA(n_components=N_SOURCES, random_state=SEED, whiten=True)
S_hat_pca = pca_sep.fit_transform(X_obs.T).T   # (sources, samples)
print("Done.")

In [ ]:
# ── Helper: align recovered sources to true sources ───────────────────────────
def find_best_permutation(
    S_true: np.ndarray,
    S_recovered: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Align recovered ICA sources to true sources by best permutation and sign.

    ICA has three fundamental ambiguities:
        1. Permutation: components may appear in any order.
        2. Sign: each component can be negated (flipped) arbitrarily.
        3. Scale: each component can be scaled (resolved by whitening).

    We enumerate all k! permutations and select the one that maximises
    the mean absolute Pearson correlation with the true sources.

    Args:
        S_true: True source signals, shape (n_sources, n_samples).
        S_recovered: Estimated sources (any order/sign), shape (n_sources, n_samples).

    Returns:
        Tuple of:
            - S_aligned: Permuted and sign-corrected recovered signals,
              shape (n_sources, n_samples).
            - corr_per_source: Absolute Pearson correlation per matched source.
    """
    k = S_true.shape[0]
    # Build absolute correlation matrix
    all_corr = np.corrcoef(S_true, S_recovered)   # (2k, 2k)
    corr_matrix = np.abs(all_corr[:k, k:])        # (k, k)

    best_mean  = -1.0
    best_perm: tuple[int, ...] = tuple(range(k))

    for perm in itertools.permutations(range(k)):
        mean_corr = float(np.mean([corr_matrix[i, perm[i]] for i in range(k)]))
        if mean_corr > best_mean:
            best_mean = mean_corr
            best_perm = perm

    # Apply best permutation
    S_aligned = S_recovered[list(best_perm), :].copy()

    # Fix sign ambiguity: negate component if its correlation with truth is negative
    corr_per_source = np.zeros(k)
    for i in range(k):
        raw_corr = float(np.corrcoef(S_true[i], S_aligned[i])[0, 1])
        if raw_corr < 0:
            S_aligned[i] = -S_aligned[i]
        corr_per_source[i] = abs(raw_corr)

    return S_aligned, corr_per_source


# Align all recovered signals
S_scratch_aligned, corr_scratch = find_best_permutation(S_true, S_hat_scratch)
S_sklearn_aligned, corr_sklearn = find_best_permutation(S_true, S_hat_sklearn)
S_pca_aligned,     corr_pca     = find_best_permutation(S_true, S_hat_pca)

print(f"From-scratch FastICA — per-source |corr|: {corr_scratch.round(4)}  mean: {corr_scratch.mean():.4f}")
print(f"sklearn FastICA      — per-source |corr|: {corr_sklearn.round(4)}  mean: {corr_sklearn.mean():.4f}")
print(f"PCA whitening        — per-source |corr|: {corr_pca.round(4)}     mean: {corr_pca.mean():.4f}")

In [ ]:
# ── Side-by-side visualisation: true / mixed / ICA / PCA ──────────────────────
t_plot_len = 500   # show first 1 second

fig = plt.figure(figsize=(16, 8))
gs  = gridspec.GridSpec(N_SOURCES, 4, figure=fig, wspace=0.05, hspace=0.6)
col_titles = ["True sources", "Mixed observations",
              "FastICA (scratch)", "PCA (decorrelation only)"]
all_signals = [S_true, X_obs, S_scratch_aligned, S_pca_aligned]
col_colors  = ["#1f77b4", "gray", "#2ca02c", "#d62728"]

for col, (data, color, title) in enumerate(zip(all_signals, col_colors, col_titles)):
    for row in range(N_SOURCES):
        ax = fig.add_subplot(gs[row, col])
        ax.plot(t_signal[:t_plot_len], data[row, :t_plot_len],
                color=color, linewidth=0.9)
        ax.set_ylim(-3.5, 3.5)
        ax.grid(True, alpha=0.25)
        if col == 0:
            ax.set_ylabel(SOURCE_NAMES[row], fontsize=8)
        if row == 0:
            ax.set_title(title, fontsize=9)
        if row == N_SOURCES - 1:
            ax.set_xlabel("Time (s)", fontsize=8)

plt.suptitle("Cocktail Party — ICA recovers sources; PCA only decorrelates", fontsize=12)
plt.show()

---
## Part 4 — Evaluation & Analysis

In [ ]:
# ── Kurtosis analysis of each signal type ─────────────────────────────────────
def compute_kurtosis(x: np.ndarray) -> float:
    """Compute excess kurtosis (kurtosis minus 3) of a 1-D signal.

    Excess kurtosis = 0 for Gaussian, > 0 for super-Gaussian (heavy tails,
    e.g. speech, spikes), < 0 for sub-Gaussian (bounded, platykurtic).

    Args:
        x: 1-D array of samples.

    Returns:
        Excess kurtosis value.
    """
    x_std = (x - x.mean()) / (x.std() + 1e-12)
    return float(np.mean(x_std ** 4) - 3.0)


# Compute kurtosis for every signal in each stage
signal_groups: list[tuple[str, np.ndarray]] = [
    ("True sources",       S_true),
    ("Mixed observations", X_obs),
    ("ICA recovered",      S_scratch_aligned),
    ("PCA recovered",      S_pca_aligned),
]

print(f"{'Signal group':25s}   {'s₁ kurt':>10}  {'s₂ kurt':>10}  {'s₃ kurt':>10}")
print("-" * 65)
for group_name, signals in signal_groups:
    kurts = [compute_kurtosis(signals[i]) for i in range(N_SOURCES)]
    print(f"{group_name:25s}   {kurts[0]:>10.4f}  {kurts[1]:>10.4f}  {kurts[2]:>10.4f}")

# Gaussian reference
gauss_ref = np.random.RandomState(SEED).randn(N_SAMPLES)
print(f"\nGaussian reference:             kurtosis = {compute_kurtosis(gauss_ref):.4f}  (≈ 0)")
print("\nInterpretation:")
print("  Sine / sawtooth: sub-Gaussian (uniform-like, bounded) → excess kurtosis < 0")
print("  Square wave:     super-Gaussian (bimodal spikes)       → excess kurtosis > 0")
print("  Mixtures:        kurtosis pushed toward 0 (central limit theorem effect)")

In [ ]:
# ── Higher-order independence test ────────────────────────────────────────────
# Statistical independence requires E[f(X)g(Y)] = E[f(X)] E[g(Y)] for ALL
# functions f, g.  Pearson correlation only tests the case f(x)=x, g(y)=y.
# A stronger practical test checks cross-moments for higher powers.

def higher_order_independence_test(
    signals: np.ndarray,
    signal_names: list[str],
    powers: list[int] | None = None,
) -> pd.DataFrame:
    """Test statistical independence via cross-moments E[x_i^p * x_j^p].

    For truly independent signals:
        E[x_i^p * x_j^p] ≈ E[x_i^p] * E[x_j^p]  for i ≠ j.
    The 'violation' measures the absolute difference.  Large violations
    indicate residual statistical dependence not captured by correlation.

    Args:
        signals: Signal matrix of shape (n_signals, n_samples).
        signal_names: Label for each row of signals.
        powers: Exponents to test. Defaults to [1, 2, 3].

    Returns:
        DataFrame with cross-moment violations for each pair and power.
    """
    if powers is None:
        powers = [1, 2, 3]
    n_sig = signals.shape[0]
    rows: list[dict] = []
    for power in powers:
        for i in range(n_sig):
            for j in range(i + 1, n_sig):
                xi_p = signals[i] ** power
                xj_p = signals[j] ** power
                cross_moment     = float(np.mean(xi_p * xj_p))
                expected_product = float(np.mean(xi_p) * np.mean(xj_p))
                violation        = abs(cross_moment - expected_product)
                rows.append({
                    "Pair":            f"{signal_names[i]} × {signal_names[j]}",
                    "Power p":         power,
                    "E[xi^p·xj^p]":   round(cross_moment, 6),
                    "E[xi^p]·E[xj^p]": round(expected_product, 6),
                    "Violation":       round(violation, 6),
                })
    return pd.DataFrame(rows)


src_names = ["s1", "s2", "s3"]
mix_names = ["x1", "x2", "x3"]
ica_names = ["s1_hat", "s2_hat", "s3_hat"]

df_sources  = higher_order_independence_test(S_true,             src_names)
df_mixtures = higher_order_independence_test(X_obs,              mix_names)
df_ica      = higher_order_independence_test(S_scratch_aligned,  ica_names)

mean_viol_src = df_sources["Violation"].mean()
mean_viol_mix = df_mixtures["Violation"].mean()
mean_viol_ica = df_ica["Violation"].mean()

print("Higher-order independence test — mean cross-moment violation:")
print(f"  True sources:       {mean_viol_src:.6f}  (small — truly independent)")
print(f"  Mixed observations: {mean_viol_mix:.6f}  (large  — correlated)")
print(f"  ICA recovered:      {mean_viol_ica:.6f}  (should be close to true sources)")

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)
stage_info = [
    (df_sources,  "True sources",       "#1f77b4"),
    (df_mixtures, "Mixed observations", "gray"),
    (df_ica,      "ICA recovered",      "#2ca02c"),
]
for ax, (df_stage, label, _) in zip(axes, stage_info):
    for power in [1, 2, 3]:
        sub = df_stage[df_stage["Power p"] == power]
        pair_labels = [row["Pair"] for _, row in sub.iterrows()]
        ax.bar(
            [f"p={power}\n{p}" for p in pair_labels],
            sub["Violation"].values,
            alpha=0.75, label=f"p={power}",
        )
    ax.set_title(label, fontsize=9)
    ax.set_ylabel("|E[x_i^p·x_j^p] − E[x_i^p]·E[x_j^p]|", fontsize=7)
    ax.tick_params(axis="x", labelsize=7, rotation=20)
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend(fontsize=8)

plt.suptitle("Cross-moment violations: higher-order independence test", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── Amari performance index ───────────────────────────────────────────────────
def amari_performance_index(
    A_true: np.ndarray,
    W_total: np.ndarray,
) -> float:
    """Compute the Amari performance index for an estimated unmixing matrix.

    Measures how close the global matrix G = W_total @ A_true is to a
    scaled permutation matrix (which would represent perfect separation).

    Formula (Amari et al., 1996):
        API = (1 / (2k(k-1))) * sum_i [
            (sum_j |G_ij|/max_l|G_il|) - 1 +
            (sum_j |G_ij|/max_l|G_lj|) - 1
        ]

    Args:
        A_true: True mixing matrix, shape (k, k).
        W_total: Total unmixing matrix W @ V, shape (k, k).

    Returns:
        API in [0, 1]. Smaller is better (0 = perfect separation).
    """
    G = np.abs(W_total @ A_true)           # global matrix (should be permutation)
    k = G.shape[0]

    row_max = np.maximum(G.max(axis=1, keepdims=True), 1e-15)
    col_max = np.maximum(G.max(axis=0, keepdims=True), 1e-15)

    row_term = np.sum(G / row_max, axis=1) - 1.0   # (k,)
    col_term = np.sum(G / col_max, axis=0) - 1.0   # (k,)

    api = (np.sum(row_term) + np.sum(col_term)) / (2.0 * k * (k - 1))
    return float(api)


# Total unmixing matrix: W (ICA in whitened space) @ V (whitening)
W_total_scratch = ica_scratch.components_ @ ica_scratch.whitening_matrix_  # (k, d)
api_scratch     = amari_performance_index(A_true, W_total_scratch)

# sklearn: recover total unmixing
# sklearn FastICA stores mixing_ in (features, components) orientation
A_est_sklearn   = ica_sklearn.mixing_           # (n_signals, n_components) in sklearn
W_sklearn_total = np.linalg.pinv(A_est_sklearn) # rough unmixing estimate
api_sklearn     = amari_performance_index(A_true, W_sklearn_total)

print(f"Amari Performance Index (lower = better):")
print(f"  From-scratch FastICA: {api_scratch:.6f}")
print(f"  sklearn FastICA:      {api_sklearn:.6f}")
print(f"  Random baseline (identity unmixing): {amari_performance_index(A_true, np.eye(N_SOURCES)):.6f}")

In [ ]:
# ── Negentropy: measuring non-Gaussianity numerically ─────────────────────────
# Negentropy J(x) = H(ν) - H(x) (Gaussian entropy minus signal entropy).
# J(x) = 0 iff x is Gaussian; J(x) > 0 otherwise.
# We approximate J(x) using the contrast function (Hyvärinen 1998):
#   J(x) ≈ [E{G(x)} - E{G(ν)}]^2  where ν ~ N(0,1)
# This approximation is accurate when x is already nearly whitened.

def compute_negentropy_logcosh(
    x: np.ndarray,
    alpha: float = 1.0,
    n_gauss_ref: int = 50_000,
    random_state: int = SEED,
) -> float:
    """Approximate negentropy using the logcosh contrast function.

    J(x) ≈ [E{log(cosh(alpha*x))} - E{log(cosh(alpha*ν))}]^2
    where ν ~ N(0,1) is a standard Gaussian reference.

    Args:
        x: 1-D signal (should be standardised to unit variance).
        alpha: Scale parameter for the logcosh contrast.
        n_gauss_ref: Number of samples for the Gaussian reference expectation.
        random_state: Seed for the Gaussian reference.

    Returns:
        Approximate negentropy value (non-negative float).
    """
    rng_ref    = np.random.RandomState(random_state)
    nu_ref     = rng_ref.randn(n_gauss_ref)
    E_G_gauss  = float(np.mean(np.log(np.cosh(alpha * nu_ref))))
    E_G_signal = float(np.mean(np.log(np.cosh(alpha * x))))
    return float((E_G_signal - E_G_gauss) ** 2)


# Compare negentropy at each processing stage
stages_neg: list[tuple[str, np.ndarray]] = [
    ("True sources",       S_true),
    ("Mixed observations", X_obs),
    ("After whitening",    X_white),
    ("ICA recovered",      S_scratch_aligned),
    ("Gaussian reference", np.random.RandomState(SEED).randn(N_SOURCES, N_SAMPLES)),
]

neg_records: list[dict] = []
for stage_name, signals in stages_neg:
    for k in range(N_SOURCES):
        s_std = signals[k].copy()
        s_std = (s_std - s_std.mean()) / (s_std.std() + 1e-12)
        neg_val = compute_negentropy_logcosh(s_std)
        kurt_val = compute_kurtosis(s_std)
        neg_records.append({
            "Stage":       stage_name,
            "Component":   k + 1,
            "Negentropy":  round(neg_val, 6),
            "Kurtosis":    round(kurt_val, 4),
        })

neg_df = pd.DataFrame(neg_records)
print("Negentropy (logcosh approximation) and excess kurtosis at each stage:")
display(neg_df.pivot_table(
    values="Negentropy", index="Stage", columns="Component", aggfunc="first"
).round(6))

# Bar chart: mean negentropy per stage
mean_neg_by_stage = neg_df.groupby("Stage")["Negentropy"].mean().reset_index()
# Preserve logical order
stage_order = [s[0] for s in stages_neg]
mean_neg_by_stage["Stage"] = pd.Categorical(mean_neg_by_stage["Stage"], categories=stage_order)
mean_neg_by_stage = mean_neg_by_stage.sort_values("Stage")

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(mean_neg_by_stage["Stage"], mean_neg_by_stage["Negentropy"],
              color=["#1f77b4", "gray", "#9467bd", "#2ca02c", "#d62728"], alpha=0.85)
ax.set_xlabel("Processing stage")
ax.set_ylabel("Mean negentropy (logcosh approx.)")
ax.set_title("Negentropy across processing stages\n(ICA restores non-Gaussianity; Gaussian baseline = 0)")
ax.tick_params(axis="x", rotation=20)
ax.grid(True, axis="y", alpha=0.3)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1e-5,
            f"{bar.get_height():.4f}", ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ── Why Gaussian sources cannot be separated ──────────────────────────────────
# Generate two independent Gaussian sources
rng_gauss = np.random.RandomState(SEED)
S_gauss   = rng_gauss.randn(2, N_SAMPLES)       # two independent Gaussians
A_gauss   = np.array([[1.5, 0.8], [0.6, 1.2]])
X_gauss   = A_gauss @ S_gauss

# After whitening, the data distribution is rotationally symmetric
X_g_c, _ = center_signals(X_gauss)
X_g_white, V_g, _, _ = whiten_signals(X_g_c)

# Compute negentropy (logcosh approximation) along many directions
angles      = np.linspace(0, np.pi, 180)
negentropy_vals: list[float] = []

# Reference E[G(ν)] for Gaussian under logcosh
nu_ref = np.random.RandomState(0).randn(10_000)
E_G_gauss = float(np.mean(np.log(np.cosh(nu_ref))))

for theta in angles:
    w_theta = np.array([np.cos(theta), np.sin(theta)])
    u = w_theta @ X_g_white
    E_G_u   = float(np.mean(np.log(np.cosh(u))))
    neg_val = (E_G_u - E_G_gauss) ** 2
    negentropy_vals.append(neg_val)

# Do the same for non-Gaussian source (sine wave whitened)
S_ng   = np.vstack([sine_wave(np.linspace(0, 4, N_SAMPLES), 3.0),
                    sawtooth_wave(np.linspace(0, 4, N_SAMPLES), 1.5)])
S_ng   = (S_ng - S_ng.mean(axis=1, keepdims=True)) / S_ng.std(axis=1, keepdims=True)
X_ng   = A_gauss @ S_ng
X_ng_c, _ = center_signals(X_ng)
X_ng_white, _, _, _ = whiten_signals(X_ng_c)

negentropy_ng: list[float] = []
for theta in angles:
    w_theta = np.array([np.cos(theta), np.sin(theta)])
    u = w_theta @ X_ng_white
    neg_val = (float(np.mean(np.log(np.cosh(u)))) - E_G_gauss) ** 2
    negentropy_ng.append(neg_val)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(np.degrees(angles), negentropy_vals,
        label="Gaussian sources (flat landscape)", linewidth=2, color="#d62728")
ax.plot(np.degrees(angles), negentropy_ng,
        label="Non-Gaussian sources (clear peaks)", linewidth=2, color="#1f77b4")
ax.set_xlabel("Rotation angle θ (degrees)")
ax.set_ylabel("Approximate negentropy J(w_θ)")
ax.set_title("Gaussian sources: negentropy is flat → ICA cannot pick a direction")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Gaussian sources: negentropy range = "
      f"[{min(negentropy_vals):.2e}, {max(negentropy_vals):.2e}]  (≈ flat)")
print("Non-Gaussian sources: negentropy range = "
      f"[{min(negentropy_ng):.2e}, {max(negentropy_ng):.2e}]  (clear peaks)")

In [ ]:
# ── Ablation: compare all three contrast functions ────────────────────────────
contrast_results: list[dict] = []

print("Ablation: all three contrast functions on the 3-source problem")
print("-" * 70)
for contrast_name in ["logcosh", "kurtosis", "exp"]:
    t0 = time.time()
    ica_ab = FastICA(
        n_components=N_SOURCES,
        contrast=contrast_name,
        max_iter=MAX_ITER,
        tol=TOL,
        random_state=SEED,
    )
    S_hat_ab = ica_ab.fit_transform(X_obs)
    elapsed  = time.time() - t0

    S_ab_aligned, corr_ab = find_best_permutation(S_true, S_hat_ab)
    W_total_ab = ica_ab.components_ @ ica_ab.whitening_matrix_
    api_ab     = amari_performance_index(A_true, W_total_ab)

    contrast_results.append({
        "Contrast":        contrast_name,
        "Mean |corr|": round(corr_ab.mean(), 4),
        "corr s₁":     round(float(corr_ab[0]), 4),
        "corr s₂":     round(float(corr_ab[1]), 4),
        "corr s₃":     round(float(corr_ab[2]), 4),
        "Amari API":   round(api_ab, 6),
        "Total iters": sum(ica_ab.n_iter_),
        "Time (s)":    round(elapsed, 3),
    })
    print(f"  {contrast_name:10s}  mean|corr|={corr_ab.mean():.4f}  "
          f"API={api_ab:.6f}  iters={sum(ica_ab.n_iter_)}  t={elapsed:.3f}s")

contrast_df = pd.DataFrame(contrast_results)
print()
display(contrast_df.style.highlight_max(subset=["Mean |corr|"], color="lightgreen")
                         .highlight_min(subset=["Amari API", "Total iters"], color="lightblue"))

### Convergence Analysis

The deflation algorithm extracts components sequentially. Each component's convergence history (plotted on a log scale) shows how quickly the fixed-point iteration finds a stable maximum-non-Gaussianity direction. Faster convergence (fewer iterations) typically corresponds to a stronger non-Gaussianity signal in that direction.

In [ ]:
# ── Convergence curves per component ──────────────────────────────────────────
fig, axes = plt.subplots(1, N_SOURCES, figsize=(13, 4), sharey=True)
for k, (ax, history) in enumerate(zip(axes, ica_scratch.convergence_histories_)):
    ax.semilogy(history, linewidth=2, color=colors[k])
    ax.axhline(TOL, color="red", linestyle="--", linewidth=1, label=f"tol={TOL:.0e}")
    ax.set_title(f"Component {k+1} — {ica_scratch.n_iter_[k]} iters")
    ax.set_xlabel("Iteration")
    ax.grid(True, alpha=0.3)
    if k == 0:
        ax.set_ylabel("|1 − |w_new · w_old||")
    ax.legend(fontsize=8)

plt.suptitle("FastICA convergence — from-scratch (logcosh)", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── Noise robustness study ────────────────────────────────────────────────────
noise_levels   = [0.0, 0.05, 0.1, 0.2, 0.5]
noise_results: list[dict] = []

print("Noise robustness study:")
for noise in noise_levels:
    X_noisy, _ = mix_signals(S_true, A=A_true, noise_std=noise, random_state=SEED)
    ica_noise  = FastICA(n_components=N_SOURCES, contrast="logcosh", random_state=SEED)
    S_hat_noise = ica_noise.fit_transform(X_noisy)
    S_noise_aligned, corr_noise = find_best_permutation(S_true, S_hat_noise)
    noise_results.append({
        "Noise std":    noise,
        "SNR (dB)":    round(20 * np.log10(1.0 / (noise + 1e-10)), 1) if noise > 0 else float("inf"),
        "Mean |corr|": round(corr_noise.mean(), 4),
        "Min |corr|":  round(corr_noise.min(), 4),
    })
    print(f"  noise_std={noise:.2f}  mean|corr|={corr_noise.mean():.4f}  "
          f"min|corr|={corr_noise.min():.4f}")

noise_df = pd.DataFrame(noise_results)
print()
display(noise_df)

fig, ax = plt.subplots(figsize=(8, 4))
finite_snr = [r["SNR (dB)"] for r in noise_results if r["SNR (dB)"] != float("inf")]
corr_means = [r["Mean |corr|"] for r in noise_results if r["Noise std"] > 0]
ax.plot(finite_snr, corr_means, marker="o", linewidth=2, color="steelblue")
ax.set_xlabel("SNR (dB)")
ax.set_ylabel("Mean |correlation| with true sources")
ax.set_title("FastICA noise robustness")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Final comparison summary table ────────────────────────────────────────────
# PCA Amari: use PCA loading vectors as unmixing matrix
W_pca_total = pca_sep.components_
api_pca     = amari_performance_index(A_true, W_pca_total)

# Mixing matrix reconstruction error: how close is (W @ A)^{-1} to A_true?
W_total_scratch = ica_scratch.components_ @ ica_scratch.whitening_matrix_
A_est_scratch   = np.linalg.pinv(W_total_scratch)   # estimated mixing matrix
recon_error     = float(np.linalg.norm(A_est_scratch - A_true, ord="fro") /
                        np.linalg.norm(A_true, ord="fro"))

# sklearn mixing matrix reconstruction
A_est_sklearn_pinv = np.linalg.pinv(W_sklearn_total)
recon_error_sk     = float(np.linalg.norm(A_est_sklearn_pinv - A_true, ord="fro") /
                           np.linalg.norm(A_true, ord="fro"))

print(f"Frobenius relative error in A reconstruction:")
print(f"  From-scratch: {recon_error:.4f}")
print(f"  sklearn:      {recon_error_sk:.4f}")
print(f"  (Note: permutation/sign ambiguity may inflate this — Amari API is the canonical metric)")

summary_data: list[dict] = [
    {
        "Method":         "PCA (baseline)",
        "Captures":       "2nd-order (variance)",
        "Mean |corr|":    round(corr_pca.mean(), 4),
        "Amari API":      round(api_pca, 4),
        "Separates ICs?": "No",
    },
    {
        "Method":         "From-scratch FastICA",
        "Captures":       "Higher-order (non-Gaussianity)",
        "Mean |corr|":    round(corr_scratch.mean(), 4),
        "Amari API":      round(api_scratch, 4),
        "Separates ICs?": "Yes",
    },
    {
        "Method":         "sklearn FastICA",
        "Captures":       "Higher-order (non-Gaussianity)",
        "Mean |corr|":    round(corr_sklearn.mean(), 4),
        "Amari API":      round(api_sklearn, 4),
        "Separates ICs?": "Yes",
    },
]

summary_df = pd.DataFrame(summary_data)
print("\nFinal comparison:")
display(summary_df.style.highlight_max(subset=["Mean |corr|"], color="lightgreen")
                        .highlight_min(subset=["Amari API"], color="lightblue"))

### Joint Distribution: Verifying Independence\n\nStatistically independent variables have a joint density that factors: $p(s_1, s_2) = p(s_1)\\,p(s_2)$. Visually, this produces axis-aligned scatter plots — no diagonal or curved structure. Correlated mixtures show off-axis density; after ICA the recovered components should approximately restore the axis-aligned structure of the original sources.

In [ ]:
# ── Joint distribution: independence before vs after ICA ─────────────────────
# Independent sources: joint = product of marginals (axes-aligned)
# Correlated mixtures: joint shows off-diagonal density
# After ICA: should recover approximately independent structure

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
titles_joint = ["True sources (independent)",
                "Mixed observations (correlated)",
                "ICA recovered (approx. independent)"]
pairs = [(0, 1), (0, 1), (0, 1)]   # always plot first two components

for ax, data, title in zip(axes, [S_true, X_obs, S_scratch_aligned], titles_joint):
    # Standardise for fair visual comparison
    d0 = (data[0] - data[0].mean()) / data[0].std()
    d1 = (data[1] - data[1].mean()) / data[1].std()
    ax.scatter(d0[:500], d1[:500], s=4, alpha=0.35, color="steelblue")
    ax.set_xlabel("Component 1 (standardised)")
    ax.set_ylabel("Component 2 (standardised)")
    ax.set_title(title, fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle("Joint distributions — ICA restores factored (independent) structure", fontsize=11)
plt.tight_layout()
plt.show()

---
## Part 5 — Summary & Lessons Learned

### Key Takeaways

1. **Independence ≠ uncorrelatedness.** PCA whitens data (removes second-order correlations) but leaves higher-order dependencies intact. ICA targets statistical independence — a strictly stronger criterion that requires exploiting non-Gaussianity.

2. **FastICA exploits the central limit theorem in reverse.** Any mixture of independent non-Gaussian signals is *more* Gaussian than the originals. FastICA searches for the *least* Gaussian projections of the whitened data, measuring non-Gaussianity via negentropy approximated with a contrast function ($\log\cosh$, kurtosis, or exponential).

3. **Preprocessing is critical.** Centering removes the mean (prevents the first component from capturing the constant offset) and whitening (PCA sphering) reduces the search from a full $(d^2)$-parameter linear transform to an orthogonal rotation, dramatically speeding up convergence.

4. **Gaussian sources are fundamentally inseparable.** When sources are Gaussian, the negentropy landscape after whitening is perfectly flat — any rotation of the whitened data is equally non-Gaussian (i.e., equally Gaussian). ICA has no criterion to prefer one rotation over another, so separation fails.

5. **ICA has three inherent ambiguities.** The sign, order, and scale of each recovered component are arbitrary. Evaluation requires resolving these by finding the best permutation and sign flip that maximises correlation with known ground-truth sources (in research settings) or the Amari performance index.

### What's Next

→ **3-06 (Gaussian Mixture Models & EM Algorithm)** extends unsupervised decomposition from finding independent axes (ICA) to fitting a generative model that assigns each sample a probability of belonging to each cluster, using the Expectation-Maximization algorithm.

### Going Further

- Hyvärinen & Oja (2000) — *Independent Component Analysis: Algorithms and Applications* — the definitive ICA survey.
- Hyvärinen (1999) — *Fast and Robust Fixed-Point Algorithms for ICA* — derives FastICA's convergence guarantees.
- Cardoso (1997) — JADE algorithm: ICA via joint diagonalisation of 4th-order cumulant tensors.
- Disentangled VAEs (β-VAE, Chen et al. 2018) — a deep-learning analogue that seeks independent latent factors.